In [1]:
!pip install transformers torch scikit-learn -q

In [2]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score

In [3]:
texts=[
    "This movie is very good",
    "I hated the product",
    "The service was poor",
    "The food was excellent",
    "The experience was terrible",
    "The staff was very friendly",
    "I am extremely satisfied with the service",
    "The quality is not good",
    "Absolutely loved the ambiance",
    "The delivery was very late",
    "I will never buy this again",
    "Highly recommended for everyone",
    "The taste was disappointing",
    "Customer support was very helpful",
    "The packaging was खराब",
    "Value for money product",
    "Not worth the price",
    "Amazing experience overall",
    "Worst purchase ever",
    "Pretty decent and acceptable"
]
labels=[1,0,0,1,0,1,1,0,1,0,0,1,0,1,0,1,0,1,0,1]
print("Number of samples:", len(texts))

Number of samples: 20


In [4]:
tokenizer=BertTokenizer.from_pretrained('bert-base-uncased')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [5]:
tokens = tokenizer.tokenize(texts[0])
print(tokens)

['this', 'movie', 'is', 'very', 'good']


In [6]:
ids = tokenizer.encode(texts[0])
print(ids)

[101, 2023, 3185, 2003, 2200, 2204, 102]


In [7]:
encodings =tokenizer(
    texts,
    truncation=True,
    padding=True,
    max_lenghts=32,
    return_tensors="pt"
)
print(encodings.keys())
print("Input IDs shape:", encodings["input_ids"].shape)
print("Attention Mask shape:", encodings["attention_mask"].shape)


KeysView({'input_ids': tensor([[  101,  2023,  3185,  2003,  2200,  2204,   102,     0,     0],
        [  101,  1045,  6283,  1996,  4031,   102,     0,     0,     0],
        [  101,  1996,  2326,  2001,  3532,   102,     0,     0,     0],
        [  101,  1996,  2833,  2001,  6581,   102,     0,     0,     0],
        [  101,  1996,  3325,  2001,  6659,   102,     0,     0,     0],
        [  101,  1996,  3095,  2001,  2200,  5379,   102,     0,     0],
        [  101,  1045,  2572,  5186,  8510,  2007,  1996,  2326,   102],
        [  101,  1996,  3737,  2003,  2025,  2204,   102,     0,     0],
        [  101,  7078,  3866,  1996,  2572, 15599,  3401,   102,     0],
        [  101,  1996,  6959,  2001,  2200,  2397,   102,     0,     0],
        [  101,  1045,  2097,  2196,  4965,  2023,  2153,   102,     0],
        [  101,  3811,  6749,  2005,  3071,   102,     0,     0,     0],
        [  101,  1996,  5510,  2001, 15640,   102,     0,     0,     0],
        [  101,  8013,  2490

In [21]:
class SentimentDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings=encodings
        self.labels=torch.tensor(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        # Fix for UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone()
        item = {key: val[idx].detach().clone() if isinstance(val[idx], torch.Tensor) else torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item

In [9]:
dataset = SentimentDataset(encodings, labels)
num_classes = len(set(labels))
print(f"Number of classes: {num_classes}")

Number of classes: 2


In [10]:
batch_size = 8
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Verify a batch
for batch in dataloader:
    print("Input IDs batch shape:", batch["input_ids"].shape)
    print("Attention Mask batch shape:", batch["attention_mask"].shape)
    print("Labels batch shape:", batch["labels"].shape)
    break

Input IDs batch shape: torch.Size([8, 9])
Attention Mask batch shape: torch.Size([8, 9])
Labels batch shape: torch.Size([8])


/tmp/ipykernel_2540/1021160209.py:10: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}


In [11]:
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=num_classes)

# Move model to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

print(f"Model loaded and moved to {device}")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded and moved to cpu


In [20]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=5e-5)

# Training loop
model.train()
for epoch in range(3):  # Train for 3 epochs
    for batch in dataloader:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device);
        attention_mask = batch['attention_mask'].to(device);
        labels = batch['labels'].to(device);
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels);
        loss = outputs.loss;
        loss.backward();
        optimizer.step();
    print(f"Epoch {epoch+1} Loss: {loss.item()}")

print("Training complete!")

/tmp/ipykernel_2540/1021160209.py:10: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}


Epoch 1 Loss: 0.0013166522840037942
Epoch 2 Loss: 0.0005416186177171767
Epoch 3 Loss: 0.00032216968247666955
Training complete!


In [19]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=5e-5)

# Training loop
model.train()
for epoch in range(3):  # Train for 3 epochs
    for batch in dataloader:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1} Loss: {loss.item()}")

print("Training complete!")

/tmp/ipykernel_2540/1021160209.py:10: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}


Epoch 1 Loss: 0.01872863620519638
Epoch 2 Loss: 0.010611250065267086
Epoch 3 Loss: 0.003018816001713276
Training complete!


In [17]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=5e-5)

# Training loop
model.train()
for epoch in range(3):  # Train for 3 epochs
    for batch in dataloader:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device);
        attention_mask = batch['attention_mask'].to(device);
        labels = batch['labels'].to(device);
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels);
        loss = outputs.loss;
        loss.backward();
        optimizer.step();
    print(f"Epoch {epoch+1} Loss: {loss.item()}")

print("Training complete!")

/tmp/ipykernel_2540/1021160209.py:10: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}


Epoch 1 Loss: 0.716659665107727
Epoch 2 Loss: 0.610714316368103
Epoch 3 Loss: 0.4514086842536926
Training complete!


In [18]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=5e-5)

# Training loop
model.train()
for epoch in range(3):  # Train for 3 epochs
    for batch in dataloader:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device);
        attention_mask = batch['attention_mask'].to(device);
        labels = batch['labels'].to(device);
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels);
        loss = outputs.loss;
        loss.backward();
        optimizer.step();
    print(f"Epoch {epoch+1} Loss: {loss.item()}")

print("Training complete!")

/tmp/ipykernel_2540/1021160209.py:10: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}


Epoch 1 Loss: 0.28279054164886475
Epoch 2 Loss: 0.07908329367637634
Epoch 3 Loss: 0.053248949348926544
Training complete!
